# 問題設定 — 回帰 vs 分類

解きたい課題を回帰・分類のどちらに落とし込むか、目的変数とタスク定義の考え方を学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/ml-basics/02-problem-framing`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# [Colab / ローカル環境用セットアップ]
# Kaggle Notebook 上では既存の /kaggle/input/titanic をそのまま使用し、
# Google Colab やローカル環境では互換スキーマの Titanic サンプルデータ（891行 / 418行）を自動配置します。
# ※ Kaggle 本番データで提出する場合は kaggle CLI (!kaggle competitions download -c titanic) をご利用ください。
import os
import numpy as np
import pandas as pd

def _ensure_titanic_data():
    target_dir = "/kaggle/input/titanic"
    if os.path.exists(os.path.join(target_dir, "train.csv")):
        return

    try:
        os.makedirs(target_dir, exist_ok=True)
        data_dir = target_dir
    except OSError:
        data_dir = os.path.abspath("./.kaggle_input/titanic")
        os.makedirs(data_dir, exist_ok=True)
        _orig_read_csv = pd.read_csv

        def _compat_read_csv(filepath_or_buffer, *args, **kwargs):
            if isinstance(filepath_or_buffer, str) and filepath_or_buffer.startswith("/kaggle/input/"):
                rel = filepath_or_buffer[len("/kaggle/input/"):]
                local_path = os.path.abspath(os.path.join("./.kaggle_input", rel))
                if os.path.exists(local_path):
                    filepath_or_buffer = local_path
            return _orig_read_csv(filepath_or_buffer, *args, **kwargs)

        pd.read_csv = _compat_read_csv

    rng = np.random.default_rng(42)
    surnames = ["Braund", "Cumings", "Heikkinen", "Futrelle", "Allen", "Moran", "McCarthy", "Palsson", "Johnson", "Nasser"]
    given_m = ["Owen", "William", "James", "Henry", "Leonard", "Timothy", "charles", "George"]
    given_f = ["Florence", "Laina", "Lily", "elisabeth", "Adele", " Marguerite", "Anna", "Mary"]

    def _make_split(n, start_id, with_target):
        pclass = rng.choice([1, 2, 3], size=n, p=[0.24, 0.21, 0.55])
        sex = rng.choice(["male", "female"], size=n, p=[0.65, 0.35])
        age = np.clip(rng.normal(29.7, 14.5, size=n), 0.42, 80.0).round(1)
        sibsp = rng.choice([0, 1, 2, 3, 4], size=n, p=[0.68, 0.23, 0.05, 0.02, 0.02])
        parch = rng.choice([0, 1, 2, 3], size=n, p=[0.76, 0.13, 0.09, 0.02])
        base_fare = np.where(pclass == 1, 60.0, np.where(pclass == 2, 20.0, 11.0))
        fare = (base_fare * rng.lognormal(0.0, 0.55, size=n)).round(2)
        embarked = rng.choice(["S", "C", "Q"], size=n, p=[0.72, 0.19, 0.09]).astype(object)

        names = []
        cabins = []
        tickets = []
        for i in range(n):
            sn = surnames[i % len(surnames)]
            if sex[i] == "male":
                title = "Master" if age[i] < 13 else "Mr"
                gn = given_m[i % len(given_m)].strip().capitalize()
            else:
                title = "Miss" if age[i] < 24 else "Mrs"
                gn = given_f[i % len(given_f)].strip().capitalize()
            names.append(f"{sn}, {title}. {gn}")
            cabins.append(f"{'ABC'[pclass[i] - 1]}{10 + (i % 90)}" if rng.random() > 0.77 else np.nan)
            tickets.append(f"PC {17500 + (i % 350)}" if pclass[i] == 1 else str(230000 + i))

        age_obj = age.astype(float)
        age_obj[rng.random(n) < 0.20] = np.nan
        embarked[rng.random(n) < 0.005] = np.nan
        if not with_target:
            fare[0] = np.nan

        data = {
            "PassengerId": np.arange(start_id, start_id + n),
            "Pclass": pclass,
            "Name": names,
            "Sex": sex,
            "Age": age_obj,
            "SibSp": sibsp,
            "Parch": parch,
            "Ticket": tickets,
            "Fare": fare,
            "Cabin": pd.Series(cabins, dtype=object),
            "Embarked": pd.Series(embarked, dtype=object),
        }
        if with_target:
            family_size = sibsp + parch + 1
            logit = (
                -1.65
                + np.where(sex == "female", 2.45, 0.0)
                + np.where(pclass == 1, 1.35, np.where(pclass == 2, 0.35, -0.55))
                + np.where(np.nan_to_num(age_obj, nan=29.0) < 12, 1.2, 0.0)
                + np.where((family_size >= 2) & (family_size <= 4), 0.75, -0.35)
            )
            prob = 1.0 / (1.0 + np.exp(-logit))
            survived = (rng.random(n) < prob).astype(int)
            ordered = {"PassengerId": data["PassengerId"], "Survived": survived}
            for k, v in data.items():
                if k != "PassengerId":
                    ordered[k] = v
            return pd.DataFrame(ordered)
        return pd.DataFrame(data)

    _make_split(891, 1, True).to_csv(os.path.join(data_dir, "train.csv"), index=False)
    _make_split(418, 892, False).to_csv(os.path.join(data_dir, "test.csv"), index=False)
    print(f"Titanic dataset ready at: {data_dir}")

_ensure_titanic_data()

## まず「何を予測するか」を決める

モデルを作る前に、**目的変数（予測したいもの）が何か**を明確にします。
目的変数の種類で、タスクの種類と使う評価指標が決まります。

## 回帰と分類

| タスク       | 目的変数       | 例                   | 代表的な評価指標                  |
| ------------ | -------------- | -------------------- | --------------------------------- |
| **回帰** | 連続値         | 住宅価格・売上・気温 | RMSE / MAE / 決定係数 |
| **分類** | クラス（離散） | 生存/死亡・犬/猫     | 正解率 / F1 / AUC     |

分類はさらに、2 クラスの**二値分類**（生存/死亡）と、3 クラス以上の**多クラス分類**に分かれます。

> **Tip:** 同じデータでも問題設定は一つではありません。たとえば「年齢」は回帰で予測することも、 「年代（10 代/20 代…）」という分類に変換することもできます。コンペでは課題の評価指標に 合わせて設定が決まっています。

## 目的変数とタスクを Titanic で確認

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/titanic/train.csv")

# Survived は 0/1 の二値 → 二値分類のタスク
df["Survived"].unique()        # array([0, 1])
df["Survived"].dtype           # int64

In [ ]:
print("unique targets:", df["Survived"].unique(), "dtype:", df["Survived"].dtype)

`Survived` は 0/1 の 2 値なので、Titanic は**二値分類**の問題です。
これに対して House Prices コンペの `SalePrice`（価格）は連続値なので**回帰**になります。

> **Note:** タスクの種類を取り違えると、評価指標もモデルも噛み合いません。 「目的変数は連続値か、クラスか」を最初に必ず確認しましょう。

## 次のステップ

次は、モデルの「本当の実力」を測るための**データ分割と汎化**の考え方を学びます。